### B-Tree:
- A tree designed to optimize data access, especially on disk-based storage systems

In [4]:
import sys

# A BTree Node
class BTreeNode:
    def __init__(self, t, leaf):
        self.keys = [0] * (2*t-1)
        self.t = t
        self.leaf = leaf
        self.n = 0
        self.C = [None] * (2*t)

    def findKey(self, k):
        idx = 0
        while idx < self.n and self.keys[idx] < k:
            idx += 1
        return idx

    # Splits the child y of this node. y must be full (2t-1 keys)
    def splitChild(self, i, y):
        z = BTreeNode(y.t, y.leaf)
        z.n = self.t - 1
        for j in range(self.t-1):
            z.keys[j] = y.keys[j + self.t]
        if not y.leaf:
            for j in range(self.t):
                z.C[j] = y.C[j + self.t]
        y.n = self.t - 1
        for j in range(self.n, i, -1):
            self.C[j+1] = self.C[j]
        self.C[i+1] = z
        for j in range(self.n-1, i-1, -1):
            self.keys[j+1] = self.keys[j]
        self.keys[i] = y.keys[self.t-1]
        self.n += 1

    def insertNonFull(self, k):
        i = self.n - 1
        if self.leaf:
            while i >= 0 and self.keys[i] > k:
                self.keys[i+1] = self.keys[i]
                i -= 1
            self.keys[i+1] = k
            self.n += 1
        else:
            while i >= 0 and self.keys[i] > k:
                i -= 1
            if self.C[i+1].n == 2*self.t-1:
                self.splitChild(i+1, self.C[i+1])
                if self.keys[i+1] < k:
                    i += 1
            self.C[i+1].insertNonFull(k)

    # Rebalances child C[idx] if it has fewer than t-1 keys
    def fill(self, idx):
        if idx!= 0 and self.C[idx-1].n >= self.t:
            child = self.C[idx]
            sib = self.C[idx-1]
            for j in range(child.n-1, -1, -1):
                child.keys[j+1] = child.keys[j]
            if not child.leaf:
                for j in range(child.n):
                    child.C[j+1] = child.C[j]
            child.keys[0] = self.keys[idx-1]
            if not child.leaf:
                child.C[0] = sib.C[sib.n]
            self.keys[idx-1] = sib.keys[sib.n-1]
            child.n += 1
            sib.n -= 1
        elif idx!= self.n and self.C[idx+1].n >= self.t:
            child = self.C[idx]
            sib = self.C[idx+1]
            child.keys[child.n] = self.keys[idx]
            if not child.leaf:
                child.C[child.n+1] = sib.C[0]
            self.keys[idx] = sib.keys[0]
            for j in range(sib.n-1):
                sib.keys[j] = sib.keys[j+1]
            if not sib.leaf:
                for j in range(sib.n):
                    sib.C[j] = sib.C[j+1]
            child.n += 1
            sib.n -= 1
        else:
            i = idx if idx!= self.n else idx-1
            child = self.C[i]
            sib = self.C[i+1]
            child.keys[self.t-1] = self.keys[i]
            for j in range(sib.n):
                child.keys[self.t + j] = sib.keys[j]
            if not child.leaf:
                for j in range(sib.n+1):
                    child.C[self.t + j] = sib.C[j]
            for j in range(self.n-i-2):
                self.keys[i + j] = self.keys[i + j + 1]
            for j in range(self.n-i-1):
                self.C[i + j] = self.C[i + j + 1]
            child.n += sib.n + 1
            self.n -= 1

    def remove(self, k):
        idx = self.findKey(k)
        if idx < self.n and self.keys[idx] == k:
            if self.leaf:
                for j in range(idx, self.n-1):
                    self.keys[j] = self.keys[j+1]
                self.n -= 1
            else:
                if self.C[idx].n >= self.t:
                    cur = self.C[idx]
                    while not cur.leaf:
                        cur = cur.C[cur.n]
                    pred = cur.keys[cur.n-1]
                    self.keys[idx] = pred
                    self.C[idx].remove(pred)
                elif self.C[idx+1].n >= self.t:
                    cur = self.C[idx+1]
                    while not cur.leaf:
                        cur = cur.C[0]
                    succ = cur.keys[0]
                    self.keys[idx] = succ
                    self.C[idx+1].remove(succ)
                else:
                    self.fill(idx)
                    self.remove(k)
        else:
            if self.leaf:
                return
            flag = (idx == self.n)
            if self.C[idx].n < self.t:
                self.fill(idx)
            if flag and idx > self.n:
                self.C[idx-1].remove(k)
            else:
                self.C[idx].remove(k)

    def traverse(self):
        for i in range(self.n):
            if not self.leaf:
                self.C[i].traverse()
            print(" {}".format(self.keys[i]), end='')
        if not self.leaf:
            self.C[self.n].traverse()

class BTree:
    def __init__(self, t):
        self.root = None
        self.t = t

    def insert(self, k):
        if self.root is None:
            self.root = BTreeNode(self.t, True)
            self.root.keys[0] = k
            self.root.n = 1
        else:
            if self.root.n == 2*self.t-1:
                s = BTreeNode(self.t, False)
                s.C[0] = self.root
                s.splitChild(0, self.root)
                i = 1 if s.keys[0] < k else 0
                s.C[i].insertNonFull(k)
                self.root = s
            else:
                self.root.insertNonFull(k)

    def remove(self, k):
        if self.root is None:
            return
        self.root.remove(k)
        if self.root.n == 0:
            self.root = None if self.root.leaf else self.root.C[0]

    def traverse(self):
        if self.root is not None:
            self.root.traverse()
        print()

In [5]:
if __name__ == '__main__':
    tree = BTree(3)
    tree.insert(10)
    tree.insert(5)
    tree.insert(15)
    tree.insert(2)
    tree.insert(7)
    tree.insert(12)
    tree.insert(20)
    print('Tree traversal:', end='')
    tree.traverse()
    tree.remove(5)
    print('After removing 5:', end='')
    tree.traverse()

Tree traversal: 2 5 7 10 12 15 20
After removing 5: 2 7 10 12 15 20
